# Calorie Burn Model — EDA & Evaluation

**Model:** Lasso Regression with Polynomial Features (degree 3)  
**Data:** `backend/app/ml/data/calories.csv` — 15,000 samples  
**Features:** Gender, Age, Height, Weight, Duration, Heart_Rate, Body_Temp  
**Target:** Calories (continuous, range ≈ 3–200)

> **Metric note:** Lasso is a regression model. Section C covers regression metrics (R², MAE, RMSE).  
> Section D adds classification-style metrics (F1, precision, false positives) by binning calorie output into Low / Medium / High brackets.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

DATA_PATH  = '../app/ml/data/calories.csv'
MODEL_PATH = '../app/ml/calorie_model.pkl'

print('Setup complete.')

---
## A — Data Loading & Overview

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Canonical column names (mirrors calorie_predictor.py aliases)
_ALIASES = {
    'Gender':     ['Gender','gender','Sex','sex'],
    'Age':        ['Age','age'],
    'Height':     ['Height','height'],
    'Weight':     ['Weight','weight'],
    'Duration':   ['Duration','duration'],
    'Heart_Rate': ['Heart_Rate','Heart Rate','heart_rate','HeartRate'],
    'Body_Temp':  ['Body_Temp','Body_Temperature','body_temp','BodyTemp'],
    'Calories':   ['Calories','calories','Calorie'],
    'User_ID':    ['User_ID','User_Id','user_id','ID'],
}

rename = {}
for canonical, aliases in _ALIASES.items():
    for a in aliases:
        if a in df_raw.columns and a != canonical:
            rename[a] = canonical
            break

df = df_raw.rename(columns=rename).copy()
if not pd.api.types.is_numeric_dtype(df['Gender']):
    df['Gender'] = (df['Gender'].str.lower() == 'male').astype(int)

FEATURES = ['Gender','Age','Height','Weight','Duration','Heart_Rate','Body_Temp']
TARGET   = 'Calories'

df = df[FEATURES + [TARGET]].dropna()
print(f'Clean shape: {df.shape}')
df.dtypes

In [ ]:
print('=== Descriptive Statistics ===')
df.describe().round(2)

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values.')

if 'User_ID' in df_raw.columns:
    dupes = df_raw['User_ID'].duplicated().sum()
    print(f'\nDuplicate User_IDs: {dupes}')

---
## B — Exploratory Data Analysis

In [ ]:
# B1 — Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df[TARGET], bins=40, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Calorie Distribution')
axes[0].set_xlabel('Calories Burned')

sns.boxplot(y=df[TARGET], color='steelblue', ax=axes[1])
axes[1].set_title('Calorie Boxplot')
axes[1].set_ylabel('Calories Burned')

skew = df[TARGET].skew()
print(f'Skewness: {skew:.3f}  ({"right-skewed" if skew > 0.5 else "left-skewed" if skew < -0.5 else "approx. symmetric"})')
plt.tight_layout()
plt.show()

In [ ]:
# B2 — Feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    if col == 'Gender':
        sns.countplot(x=df_raw['Gender'] if 'Gender' in df_raw.columns else df[col].map({1:'male',0:'female'}),
                      ax=axes[i], palette='Set2')
        axes[i].set_title('Gender')
    else:
        sns.histplot(df[col], bins=30, kde=True, ax=axes[i], color='mediumpurple')
        axes[i].set_title(col)

axes[-1].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# B3 — Gender vs Calories
gender_label = df['Gender'].map({1: 'Male', 0: 'Female'})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
gender_label.value_counts().plot.bar(ax=axes[0], color=['steelblue','salmon'], rot=0)
axes[0].set_title('Sample Count by Gender')

df.groupby(gender_label)[TARGET].mean().plot.bar(ax=axes[1], color=['steelblue','salmon'], rot=0)
axes[1].set_title('Mean Calories by Gender')
axes[1].set_ylabel('Avg Calories')

plt.tight_layout()
plt.show()

In [ ]:
# B4 — Correlation heatmap
plt.figure(figsize=(9, 7))
corr = df[FEATURES + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nTop correlates with Calories:')
print(corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).to_string())

In [ ]:
# B5 — Key scatter plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [('Duration', TARGET), ('Heart_Rate', TARGET), ('Weight', TARGET)]

for ax, (x, y) in zip(axes, pairs):
    colors = df['Gender'].map({1: 'steelblue', 0: 'salmon'})
    ax.scatter(df[x], df[y], c=colors, alpha=0.3, s=10)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'{x} vs {y}')

from matplotlib.patches import Patch
legend = [Patch(color='steelblue', label='Male'), Patch(color='salmon', label='Female')]
axes[-1].legend(handles=legend, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# B6 — Outlier detection (IQR method)
print('=== Outlier Count per Feature (IQR method) ===')
outlier_summary = {}
for col in FEATURES + [TARGET]:
    if col == 'Gender':
        continue
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_summary[col] = n_out
    print(f'  {col:12s}: {n_out:4d} outliers ({n_out/len(df)*100:.1f}%)')

---
## C — Regression Model Evaluation

In [ ]:
X = df[FEATURES].values.astype(float)
y = df[TARGET].values.astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['Gender'].values
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

model = joblib.load(MODEL_PATH)
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 0, None)  # predictions can't be negative

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'\n=== Hold-out Test Metrics (n={X_test.shape[0]}) ===')
print(f'  R²   : {r2:.4f}')
print(f'  MAE  : {mae:.3f} cal')
print(f'  RMSE : {rmse:.3f} cal')

In [ ]:
# 5-fold cross-validation on full dataset
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f'5-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold:     {np.round(cv_scores, 4)}')

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Calories')
axes[0].set_ylabel('Predicted Calories')
axes[0].set_title(f'Predicted vs Actual  (R²={r2:.3f})')
axes[0].legend()

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, s=10, color='mediumpurple')
axes[1].axhline(0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Calories')
axes[1].set_ylabel('Residual (Actual − Predicted)')
axes[1].set_title('Residuals vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution + normality test
sample_res = residuals[:5000] if len(residuals) > 5000 else residuals
stat, p = stats.shapiro(sample_res)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(residuals, bins=40, kde=True, color='coral', ax=axes[0])
axes[0].set_title(f'Residual Distribution  (Shapiro p={p:.4f})')
axes[0].set_xlabel('Residual')

stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()
print(f'Shapiro-Wilk: stat={stat:.4f}, p={p:.4f}  →  Residuals {"appear normal" if p > 0.05 else "are NOT normal (p<0.05)"}')

---
## D — Classification-style Metrics (F1, Precision, False Positives)

Calorie output binned into three classes:

| Class | Range |
|-------|-------|
| Low | < 50 cal |
| Medium | 50 – 119 cal |
| High | ≥ 120 cal |

A **false positive** for a class means the model predicted that class when the actual was different (e.g., predicted *High* when actually *Medium*).

In [ ]:
def bin_calories(arr):
    labels = np.where(arr < 50, 'Low', np.where(arr < 120, 'Medium', 'High'))
    return labels

y_true_cls = bin_calories(y_test)
y_pred_cls = bin_calories(y_pred)

class_order = ['Low', 'Medium', 'High']

print('=== Classification Report (binned calories) ===')
print(classification_report(y_true_cls, y_pred_cls, labels=class_order, zero_division=0))

In [ ]:
# Confusion matrix — off-diagonal cells are false positives/negatives
cm = confusion_matrix(y_true_cls, y_pred_cls, labels=class_order)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Calorie Bins\n(off-diagonal = misclassifications / false positives)')
plt.tight_layout()
plt.show()

# False positives per class
print('\n=== False Positives per Class ===')
for i, cls in enumerate(class_order):
    fp = cm[:, i].sum() - cm[i, i]
    print(f'  {cls:6s}: {fp} false positives (predicted {cls} when actually different)')

---
## E — Feature Importance (Lasso Coefficients)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = model.named_steps['poly']
lasso = model.named_steps['lasso']

poly_feature_names = poly.get_feature_names_out(FEATURES)
coefs = pd.Series(lasso.coef_, index=poly_feature_names)

nonzero = coefs[coefs != 0].abs().sort_values(ascending=False)
top15 = nonzero.head(15)

print(f'Non-zero coefficients: {len(nonzero)} / {len(coefs)} total polynomial features')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if coefs[f] > 0 else 'salmon' for f in top15.index]
top15[::-1].plot.barh(ax=ax, color=colors[::-1])
ax.set_title('Top 15 Non-Zero Lasso Coefficients (absolute value)\nBlue = positive effect, Red = negative effect')
ax.set_xlabel('|Coefficient|')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()